# Week 3: Prompts as Engineering Artifacts

This notebook evaluates two version-controlled support-ticket prompts over a 10-case test suite using exact-match accuracy and semantic similarity. When opened through GitHub-to-Colab, the first code cell clones the repository so the notebook reads the actual prompt files from `week-03/prompts/`.

In Colab, store the Gemini key in **Secrets** as `GEMINI_API_KEY`. Locally, set the same name as an environment variable.

Dependencies: `sentence-transformers`, `openai`, and `pandas`.


In [45]:
# Colab repository setup
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/mschemerii/cosc-650-applied-llm-systems.git'
REPO_DIR = pathlib.Path('/content/cosc-650-applied-llm-systems')
ASSIGNMENT_BRANCH = 'week3assignment'

if 'google.colab' in sys.modules:
    branch_exists = subprocess.run(
        ['git', 'ls-remote', '--exit-code', '--heads', REPO_URL, ASSIGNMENT_BRANCH],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode == 0
    branch = ASSIGNMENT_BRANCH if branch_exists else 'main'

    if not REPO_DIR.exists():
        subprocess.run(
            ['git', 'clone', '--branch', branch, '--single-branch', REPO_URL, str(REPO_DIR)],
            check=True
        )
        print(f'cloned branch: {branch}')
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', branch], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', branch], check=True)
        print(f'updated branch: {branch}')

    os.chdir(REPO_DIR / 'week-03')
    print('working directory:', pathlib.Path.cwd())
else:
    print('non-Colab environment; using current checkout:', pathlib.Path.cwd())


updated branch: week3assignment
working directory: /content/cosc-650-applied-llm-systems/week-03


In [46]:
import json, time, hashlib

MODEL_NAME = 'gemini-3.5-flash-lite'
REQUEST_DELAY_SECONDS = 5

def get_gemini_key():
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            key = userdata.get('GEMINI_API_KEY')
            if key:
                return key, 'Colab Secrets'
        except Exception as exc:
            print('Colab secret unavailable:', type(exc).__name__)
        return None, 'Colab Secrets'

    key = os.environ.get('GEMINI_API_KEY')
    return (key, 'environment variable') if key else (None, 'environment variable')

GEMINI_API_KEY, KEY_SOURCE = get_gemini_key()

def gemini_chat(messages, model=MODEL_NAME, max_retries=5, **kw):
    if not GEMINI_API_KEY:
        return None
    from openai import OpenAI
    client = OpenAI(
        api_key=GEMINI_API_KEY,
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
    )
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model=model, messages=messages, **kw)
            time.sleep(REQUEST_DELAY_SECONDS)
            return response.choices[0].message.content
        except Exception as exc:
            if '429' not in str(exc) or attempt == max_retries - 1:
                raise
            wait_seconds = 12 * (attempt + 1)
            print(f'Rate limit reached; retrying in {wait_seconds}s...')
            time.sleep(wait_seconds)

LIVE = bool(GEMINI_API_KEY)
print('Gemini key source:', KEY_SOURCE)
print('Gemini model:', MODEL_NAME)
print('live model calls:', LIVE, '(fixtures used when False)')


Gemini key source: Colab Secrets
Gemini model: gemini-3.5-flash-lite
live model calls: True (fixtures used when False)


In [47]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())

def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)

print('metrics ready (exact-match + semantic)')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite

The prompt versions are stored as repository files under `week-03/prompts/`. In Colab, the repository setup cell clones the repo first so these files are read directly from the checked-out branch.


In [48]:
def locate_prompt_dir():
    candidates = [pathlib.Path('prompts'), pathlib.Path('week-03') / 'prompts']
    for candidate in candidates:
        if (candidate / 'support_ticket_v1.txt').exists() and (candidate / 'support_ticket_v2.txt').exists():
            return candidate
    raise FileNotFoundError(
        'Prompt files not found. Run the Colab repository setup cell first, or run locally from the repo root/week-03 directory.'
    )

PROMPT_DIR = locate_prompt_dir()
PROMPT_V1 = (PROMPT_DIR / 'support_ticket_v1.txt').read_text(encoding='utf-8')
PROMPT_V2 = (PROMPT_DIR / 'support_ticket_v2.txt').read_text(encoding='utf-8')
print('prompt directory:', PROMPT_DIR.resolve())

tests = [
  {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
  {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
  {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
  {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
  {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
  {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
  {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
  {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
  {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
  {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
]
print('prompt versions:', 2, '| test cases:', len(tests))


prompt directory: /content/cosc-650-applied-llm-systems/week-03/prompts
prompt versions: 2 | test cases: 10


In [49]:
FIX = {
  'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
  'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
}

CACHE_PATH = pathlib.Path('.week3_response_cache.json')
CACHE = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}

def cache_key(version, prompt, ticket):
    payload = json.dumps({
        'model': MODEL_NAME,
        'version': version,
        'prompt': prompt,
        'ticket': ticket
    }, sort_keys=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def parse_json_response(txt):
    cleaned = (txt or '').strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        if cleaned.lower().startswith('json'):
            cleaned = cleaned[4:].lstrip()
    try:
        data = json.loads(cleaned)
        return data.get('category',''), data.get('rationale','')
    except Exception:
        return '', txt or ''

def run_case(version, prompt, t):
    if not LIVE:
        return FIX[version][t['id']]

    key = cache_key(version, prompt, t['ticket'])
    if key in CACHE:
        cached = CACHE[key]
        return cached['category'], cached['rationale']

    txt = gemini_chat([
        {'role':'system','content': prompt},
        {'role':'user','content': 'Ticket: ' + t['ticket']}
    ], temperature=0)

    category, rationale = parse_json_response(txt)
    CACHE[key] = {
        'model': MODEL_NAME,
        'category': category,
        'rationale': rationale,
        'raw': txt
    }
    CACHE_PATH.write_text(json.dumps(CACHE, indent=2), encoding='utf-8')
    return category, rationale

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({
            'id': t['id'],
            'exact': exact_match(t['cat'], cat),
            'sem': semantic_sim(t['why'], why),
            'got': cat,
            'rationale': why
        })
    acc = sum(r['exact'] for r in rows) / len(rows)
    mean_sem = sum(r['sem'] for r in rows) / len(rows)
    return acc, mean_sem, rows

acc1, sem1, r1 = score('v1', PROMPT_V1)
acc2, sem2, r2 = score('v2', PROMPT_V2)

print(f'v1 exact-match {acc1:.0%} | mean semantic {sem1:.3f}')
print(f'v2 exact-match {acc2:.0%} | mean semantic {sem2:.3f}')


v1 exact-match 90% | mean semantic 0.519
v2 exact-match 90% | mean semantic 0.511


## Part 3 and 4: compare every test case

The table below shows both prompt versions side by side for all 10 cases, including category accuracy and the change in rationale semantic similarity. This makes the strongest improvement and regression visible even when exact-match accuracy does not change.


In [50]:
import pandas as pd

comparison_rows = []
for t in tests:
    row1 = next(r for r in r1 if r['id'] == t['id'])
    row2 = next(r for r in r2 if r['id'] == t['id'])
    comparison_rows.append({
        'ID': t['id'],
        'Expected': t['cat'],
        'v1 Category': row1['got'],
        'v2 Category': row2['got'],
        'v1 Exact': int(row1['exact']),
        'v2 Exact': int(row2['exact']),
        'v1 Semantic': row1['sem'],
        'v2 Semantic': row2['sem'],
        'Delta Semantic': round(row2['sem'] - row1['sem'], 3),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

best = max(comparison_rows, key=lambda r: r['Delta Semantic'])
worst = min(comparison_rows, key=lambda r: r['Delta Semantic'])
print(
    f"Strongest semantic improvement: #{best['ID']} "
    f"{best['v1 Semantic']:.3f} -> {best['v2 Semantic']:.3f} "
    f"(delta {best['Delta Semantic']:+.3f})"
)
print(
    f"Strongest semantic regression: #{worst['ID']} "
    f"{worst['v1 Semantic']:.3f} -> {worst['v2 Semantic']:.3f} "
    f"(delta {worst['Delta Semantic']:+.3f})"
)

changed = [r for r in comparison_rows if r['v1 Exact'] != r['v2 Exact']]
if changed:
    print('Exact-match category changes:')
    for row in changed:
        verdict = 'IMPROVED' if row['v2 Exact'] > row['v1 Exact'] else 'REGRESSED'
        print(f"#{row['ID']} {verdict}: {row['v1 Category']} -> {row['v2 Category']}")
else:
    print('No exact-match category changes between v1 and v2.')


,ID,Expected,v1 Category,v2 Category,v1 Exact,v2 Exact,v1 Semantic,v2 Semantic,Delta Semantic
0,1,billing,billing,billing,1,1,0.719,0.710,-0.009
1,2,technical,technical,technical,1,1,0.616,0.570,-0.046
2,3,account,technical,technical,0,0,0.172,0.114,-0.058
3,4,shipping,shipping,shipping,1,1,0.540,0.364,-0.176
4,5,account,account,account,1,1,0.291,0.433,0.142
5,6,account,account,account,1,1,0.626,0.678,0.052
6,7,shipping,shipping,shipping,1,1,0.467,0.467,0.000
7,8,billing,billing,billing,1,1,0.662,0.628,-0.034
8,9,technical,technical,technical,1,1,0.479,0.513,0.034
9,10,account,account,account,1,1,0.618,0.634,0.016


Strongest semantic improvement: #5 0.291 -> 0.433 (delta +0.142)
Strongest semantic regression: #4 0.540 -> 0.364 (delta -0.176)
No exact-match category changes between v1 and v2.


## Part 5: Submit

Run the suite with `GEMINI_API_KEY` available, save the executed notebook, record the measured v1/v2 results in `research-note.md`, and open a pull request containing the notebook, versioned prompt files, results summary, and linked research note.
